<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:




```python
import json
import pennylane as qml
import pennylane.numpy as np
from pennylane import qchem

symbols = ["H", "H", "H"]

def h3_ground_energy(bond_length):
    """
    Uses VQE to calculate the ground energy of the H3+ molecule with the given bond length.
    
    Args:
        - bond_length(float): The bond length of the H3+ molecule modelled as an
        equilateral triangle.
    Returns:
        - Union[float, np.tensor, np.array]: A float-like output containing the ground 
        state of the H3+ molecule with the given bond length.
    """
    # Define coordinates for the equilateral triangle
    geometry = np.array([
        [0.0, 0.0, 0.0],
        [bond_length, 0.0, 0.0],
        [bond_length / 2, (bond_length * np.sqrt(3)) / 2, 0.0]
    ])

    # Build the molecular Hamiltonian
    h, qubits = qchem.molecular_hamiltonian(
        symbols=symbols,
        coordinates=geometry,
        charge=1,  # H3+ has a +1 charge
        active_electrons=2,  # 3 H atoms: 3 electrons total - 1 due to +1 charge = 2
        active_orbitals=3,   # 3 orbitals for the minimal basis set
        method="pyscf"
    )

    # Use the Strongly Entangling Layers circuit as ansatz
    wires = list(range(qubits))
    dev = qml.device("default.qubit", wires=qubits)

    # Number of layers in the ansatz
    n_layers = 6  # Increased depth for better expressibility

    # Define the ansatz
    def ansatz(params):
        qml.StronglyEntanglingLayers(
            weights=params,
            wires=wires
        )

    # Define the cost function using VQE
    @qml.qnode(dev, interface="autograd")
    def cost_function(params):
        ansatz(params)
        return qml.expval(h)

    # Initialize parameters for the ansatz
    np.random.seed(42)
    shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=qubits)
    params = np.random.uniform(low=-np.pi, high=np.pi, size=shape)

    # Use the Adam Optimizer for better convergence
    opt = qml.AdamOptimizer(stepsize=0.05)

    # Optimize the parameters
    prev_energy = float('inf')
    for _ in range(400):  # Increased number of iterations
        params = opt.step(lambda p: cost_function(p), params)
        energy = cost_function(params)
        if abs(energy - prev_energy) < 1e-6:
            break
        prev_energy = energy

    # Return the optimized energy
    return cost_function(params)


# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = h3_ground_energy(ins)
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, atol=1e-4), "Not the correct ground energy"


# These are the public test cases
test_cases = [
    ('1.5', '-1.232574'),
    ('0.8', '-0.3770325')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
        except AssertionError as e:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
        else:
            print("Correct!")


SyntaxError: invalid syntax (790137128.py, line 1)